# Explore a real ingested Discogs catalog

Real data, not a fixture. Reads the actual Parquet dataset a completed
`scripts/run-ingest.sh` run produced, mounted read-only into this container
and the local `dask-worker` containers at `/data/discogs` (see
`docker-compose.jupyter.yml` / `docker-stack.dask.yml`'s `DISCOGS_PROCESSED_DIR`
bind mount). Uses the exact same convention as
`scripts/profile-discogs-dataset.sh`: a `SNAPSHOT` env var selecting
`snapshot=<date>` under the mounted root.

**If you haven't run a real ingest yet**, the next cell fails with a clear
message -- see `docs/DISCOGS_INGESTION.md`. This notebook never silently
substitutes synthetic data and presents it as real.

**Former limitation, now addressed:** a second Dask worker with no
`/data/discogs` bind mount of its own (e.g. the optional build node,
`docker-compose.dask-worker-remote.yml`) used to fail on any partition
scheduled there. Run the "read via a worker-local cache or the catalog-data
HTTP layer" cell below when that worker has `CATALOG_DATA_DIR` (ADR 0025) or
`CATALOG_DATA_URL` (ADR 0024) set -- it overwrites `releases_glob`/
`credits_glob` for every cell below it, local-mount or not.


In [ ]:
import os
from pathlib import Path

PROCESSED_ROOT = Path("/data/discogs")
snapshot = os.environ.get("SNAPSHOT")
if not snapshot:
    raise RuntimeError(
        "Set SNAPSHOT (e.g. SNAPSHOT=20260601) as a Jupyter kernel/container "
        "env var, matching a completed scripts/run-ingest.sh run. See "
        "docs/DISCOGS_INGESTION.md."
    )

dataset_root = PROCESSED_ROOT / f"snapshot={snapshot}"
releases_glob = str(dataset_root / "table=releases" / "*.parquet")
credits_glob = str(dataset_root / "table=credits" / "*.parquet")

if not (dataset_root / "table=releases").is_dir():
    raise RuntimeError(
        f"No dataset at {dataset_root} -- confirm DISCOGS_PROCESSED_DIR in "
        "local/dask.env points at the right host path, and that "
        f"snapshot={snapshot} has actually completed (docs/DISCOGS_INGESTION.md)."
    )
print(f"Using real dataset at {dataset_root}")

## Optional: read via a worker-local cache or the catalog-data HTTP layer instead

The cell above assumes the bind mount at `/data/discogs` (this container and any
LOCAL Dask worker). A worker with no such mount -- e.g. the optional build node,
or any host relying on ADR 0024/0025's data-access layer instead -- needs a
different source. `dataset_locator.resolve_dataset` picks one for you, in order:
a validated local dataset cache (`CATALOG_DATA_DIR`, ADR 0025 --
`dataset_fetch.py` / `replicate-dataset-x86.yml`), then the catalog-data HTTP
server (`CATALOG_DATA_URL`, ADR 0024), then a clear error naming both. Run this
cell instead of the one above when either env var is set; it's a no-op fallback
to the bind-mount cell's globs otherwise.


In [ ]:
import os

if os.environ.get("CATALOG_DATA_DIR") or os.environ.get("CATALOG_DATA_URL"):
    from networked_players_catalog.discogs.dataset_locator import (
        dataset_file_urls,
        resolve_dataset,
    )

    source = resolve_dataset("discogs", snapshot)
    print(f"Resolved dataset source: {source.kind} -> {source.base}")

    if source.kind == "local":
        dataset_root = Path(source.base)
        releases_glob = str(dataset_root / "table=releases" / "*.parquet")
        credits_glob = str(dataset_root / "table=credits" / "*.parquet")
    else:  # "http" -- dd.read_parquet takes an explicit URL list, not a glob
        releases_glob = [f.url for f in dataset_file_urls(source.base, table="releases")]
        credits_glob = [f.url for f in dataset_file_urls(source.base, table="credits")]
else:
    print("Neither CATALOG_DATA_DIR nor CATALOG_DATA_URL is set -- keeping bind-mount globs.")

In [ ]:
import duckdb

# Local, non-distributed profiling for comparison -- same pattern as
# scripts/profile-discogs-dataset.sh.
con = duckdb.connect()
con.execute(f"CREATE VIEW releases AS SELECT * FROM '{releases_glob}'")
con.execute(f"CREATE VIEW credits AS SELECT * FROM '{credits_glob}'")
con.sql("SELECT count(*) AS release_rows FROM releases").show()
con.sql("SELECT country, count(*) n FROM releases GROUP BY 1 ORDER BY 2 DESC LIMIT 10").show()

In [ ]:
import dask.dataframe as dd
from dask.distributed import Client

client = Client("tcp://dask-scheduler:8786")

# Real distributed read -- each worker opens the partitions it's assigned
# directly from the shared /data/discogs mount, no data movement through
# this notebook's own kernel. Real bug found and fixed against the actual
# completed ingest (3,839 real part files per table): dd.read_parquet()
# without an explicit blocksize aggregated ALL of them into a SINGLE
# ~3.9GB partition for credits (confirmed via .npartitions == 1 below) --
# a real distributed join then has to shuffle that whole table through one
# worker at once, which is what OOM-killed it. blocksize="64MB" forces
# many smaller partitions instead, matching how a real production job
# would actually want to read this dataset.
releases = dd.read_parquet(releases_glob, blocksize="64MB")
credits = dd.read_parquet(credits_glob, blocksize="64MB")
print(f"releases partitions: {releases.npartitions}, credits partitions: {credits.npartitions}")

In [ ]:
# A genuinely distributed computation: releases per country, then a
# releases<->credits join -- cross-check the count against the DuckDB view
# above as a correctness sanity check, not just a speed demo.
by_country = releases.groupby("country").size().compute().sort_values(ascending=False)
print(by_country.head(10))

joined_count = releases.merge(credits, on="release_id", how="inner").shape[0].compute()
duckdb_joined_count = con.sql(
    "SELECT count(*) FROM releases r JOIN credits c USING (release_id)"
).fetchone()[0]
assert joined_count == duckdb_joined_count, (joined_count, duckdb_joined_count)
print(f"releases-credits join row count (Dask, cross-checked against DuckDB): {joined_count}")